In [4]:
!pip install streamlit pyngrok requests pandas scikit-learn matplotlib


In [5]:
%%writefile app.py
import streamlit as st
import requests
import pandas as pd
from sklearn.linear_model import LinearRegression
import datetime
import matplotlib.pyplot as plt

# ==============================
# CONFIG / API
# ==============================
API_KEY = "c747454a73a94e41afd172845250911"
BASE = "http://api.weatherapi.com/v1"
st.set_page_config(page_title="Weather Dashboard", layout="wide")

# GLOBAL STYLE (DARK + GLASS UI)
st.markdown("""
<style>
body {
    background-color: #0b1221;
}
[data-testid="stAppViewContainer"] {
    background: linear-gradient(160deg, #041b33 0%, #0b1221 50%, #0d203a 100%);
}
.card {
    background: rgba(255,255,255,0.08);
    padding: 22px;
    border-radius: 16px;
    color: #fff;
    box-shadow: 0 4px 15px rgba(0,0,0,.35);
    backdrop-filter: blur(7px);
}
</style>
""", unsafe_allow_html=True)


# ==============================
# DATA FETCHING FUNCTIONS
# ==============================
def get_current_weather(location):
    url = f"{BASE}/current.json?key={API_KEY}&q={location}&aqi=no"
    return requests.get(url).json()

def get_forecast(location, days=5):
    url = f"{BASE}/forecast.json?key={API_KEY}&q={location}&days={days}"
    return requests.get(url).json()

def get_history(location, days=30):
    data = []
    for i in range(1, days+1):
        dt = (datetime.datetime.utcnow() - datetime.timedelta(days=i)).strftime("%Y-%m-%d")
        try:
            res = requests.get(f"{BASE}/history.json?key={API_KEY}&q={location}&dt={dt}").json()
            d = res["forecast"]["forecastday"][0]["day"]
            data.append([dt, d["avgtemp_c"], d["maxtemp_c"], d["mintemp_c"]])
        except:
            pass
    return pd.DataFrame(data, columns=["date","avg","max","min"])


# ==============================
# DASHBOARD
# ==============================
st.markdown("<h1 style='color:white;text-align:center;'>🌤️ Weather Forecast Dashboard</h1>", unsafe_allow_html=True)

city = st.text_input("Enter city name (Example: London, Tokyo, Delhi)", "")

if st.button("Generate Dashboard"):

    if city.strip() == "":
        st.warning("Please enter a valid city")
        st.stop()

    current = get_current_weather(city)
    forecast = get_forecast(city)

    if "error" in current:
        st.error("❌ Invalid location. Try again.")
        st.stop()

    c = current["current"]

    # ------------------ WEATHER INFO CARD --------------------
    st.markdown(f"<h3 style='color:white;'>📍 {city}</h3>", unsafe_allow_html=True)
    col1, col2, col3, col4, col5, col6 = st.columns(6)

    with col1: st.markdown(f"<div class='card'><h3>{c['temp_c']} °C</h3><p>Temperature</p></div>", unsafe_allow_html=True)
    with col2: st.markdown(f"<div class='card'><h3>{c['feelslike_c']} °C</h3><p>Feels Like</p></div>", unsafe_allow_html=True)
    with col3: st.markdown(f"<div class='card'><h3>{c['humidity']}%</h3><p>Humidity</p></div>", unsafe_allow_html=True)
    with col4: st.markdown(f"<div class='card'><h3>{c['wind_kph']} km/h</h3><p>Wind Speed</p></div>", unsafe_allow_html=True)
    with col5: st.markdown(f"<div class='card'><h3>{c['pressure_mb']} mb</h3><p>Pressure</p></div>", unsafe_allow_html=True)
    with col6: st.markdown(f"<div class='card'><h3>{c['vis_km']} km</h3><p>Visibility</p></div>", unsafe_allow_html=True)

    # ------------------ 5-DAY FORECAST --------------------
    st.subheader("🗓 5-Day Forecast")

    col = st.columns(5)
    for i, d in enumerate(forecast["forecast"]["forecastday"]):
        with col[i]:
            st.markdown(
                f"""
                <div class='card'>
                <h4>{d['date']}</h4>
                {d["day"]["condition"]["text"]} <br>
                🌡 High: {d["day"]["maxtemp_c"]}°C<br>
                🌡 Low: {d["day"]["mintemp_c"]}°C<br>
                💧 Rain: {d["day"]["daily_chance_of_rain"]}%
                </div>
                """,
                unsafe_allow_html=True
            )

    # ------------------ ML PREDICTION --------------------
    st.subheader("🤖 ML Based Future Temperature Prediction")

    df = get_history(city)

    if df.empty:
        st.warning("Your API plan doesn't allow historical data.")
        st.stop()

    model = LinearRegression()
    model.fit(df[["max","min"]], df["avg"])
    predicted = model.predict([[df["avg"].iloc[-1] + 2, df["avg"].iloc[-1] - 2]])[0]

    st.success(f"Predicted Tomorrow Avg Temperature: **{predicted:.2f} °C**")

    # ------------------ BEAUTIFUL GRAPH --------------------
    st.subheader("📈 Past 30 Days Temperature Trend (ML Training Data Used)")

    df["date"] = pd.to_datetime(df["date"])
    plt.figure(figsize=(10,4))
    plt.plot(df["date"], df["avg"], marker="o", linestyle="-", linewidth=2, markersize=6)
    plt.title("Temperature Trend", fontsize=14)
    plt.xticks(rotation=45)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    st.pyplot(plt)


Overwriting app.py


In [6]:
from pyngrok import ngrok
ngrok.set_auth_token("35JoX1sxcyCYgj7Mp0sD1u4zqDD_21PozFrqNKpFHGDP5tUpo")  # <-- paste your token here

public_url = ngrok.connect(8501)
print("🔗 OPEN THIS LINK IN BROWSER:", public_url)

!streamlit run app.py --server.port 8501 > /dev/null


🔗 OPEN THIS LINK IN BROWSER: NgrokTunnel: "https://unvaunted-celinda-unfabulously.ngrok-free.dev" -> "http://localhost:8501"
